In [3]:
import torch, os
print(torch.__version__, '| GPU:', torch.cuda.is_available())

os.makedirs('/kaggle/working/project', exist_ok=True)
%cd /kaggle/working/project
!cp -r /kaggle/input/datasets/alekomamukashvili/pet-project/src .
!cp -r /kaggle/input/datasets/alekomamukashvili/pet-project/models .
!cp -r /kaggle/input/datasets/alekomamukashvili/pet-project/reports .
!ls

2.11.0+cu128 | GPU: True
/kaggle/working/project
models	reports  src


In [6]:
import torch, json
from src.data import get_dataloaders
from src.models import build_resnet18
from src.train import fit

device = 'cuda'
torch.manual_seed(42)

train_loader, val_loader, test_loader, classes = get_dataloaders(
    root='/kaggle/working/data', batch_size=32, num_workers=0)

model = build_resnet18(num_classes=37, freeze_backbone=True)

history = fit(model, train_loader, val_loader,
              epochs=8, lr=1e-3, device=device,
              ckpt_path='models/resnet18_frozen.pth')

with open('reports/resnet18_frozen_history.json', 'w') as f:
    json.dump(history, f)

100%|██████████| 792M/792M [00:17<00:00, 45.8MB/s] 
100%|██████████| 19.2M/19.2M [00:01<00:00, 18.5MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 221MB/s]


epoch 01/8 | train loss 2.114 acc 0.527 | val loss 1.115 acc 0.768 | 26s
epoch 02/8 | train loss 0.835 acc 0.831 | val loss 0.710 acc 0.841 | 25s
epoch 03/8 | train loss 0.566 acc 0.890 | val loss 0.557 acc 0.870 | 24s
epoch 04/8 | train loss 0.454 acc 0.910 | val loss 0.514 acc 0.868 | 24s
epoch 05/8 | train loss 0.409 acc 0.915 | val loss 0.466 acc 0.887 | 24s
epoch 06/8 | train loss 0.366 acc 0.925 | val loss 0.457 acc 0.882 | 24s
epoch 07/8 | train loss 0.356 acc 0.926 | val loss 0.448 acc 0.889 | 24s
epoch 08/8 | train loss 0.340 acc 0.933 | val loss 0.437 acc 0.886 | 24s
საუკეთესო val acc: 0.889


In [8]:
import torch, json
from src.models import build_resnet18
from src.train import fit

torch.manual_seed(42)
model = build_resnet18(num_classes=37, freeze_backbone=False)

history = fit(model, train_loader, val_loader,
              epochs=8, lr=1e-4, device='cuda',
              ckpt_path='models/resnet18_finetune.pth')

with open('reports/resnet18_finetune_history.json', 'w') as f:
    json.dump(history, f)

epoch 01/8 | train loss 1.812 acc 0.615 | val loss 0.796 acc 0.845 | 29s
epoch 02/8 | train loss 0.564 acc 0.901 | val loss 0.519 acc 0.878 | 29s
epoch 03/8 | train loss 0.293 acc 0.953 | val loss 0.395 acc 0.905 | 29s
epoch 04/8 | train loss 0.190 acc 0.977 | val loss 0.363 acc 0.910 | 29s
epoch 05/8 | train loss 0.132 acc 0.986 | val loss 0.341 acc 0.912 | 29s
epoch 06/8 | train loss 0.093 acc 0.995 | val loss 0.318 acc 0.916 | 29s
epoch 07/8 | train loss 0.074 acc 0.997 | val loss 0.315 acc 0.920 | 29s
epoch 08/8 | train loss 0.067 acc 0.998 | val loss 0.313 acc 0.921 | 29s
საუკეთესო val acc: 0.921


In [9]:
!cd /kaggle/working/project && zip -r /kaggle/working/results_tl.zip models reports -x "*/__pycache__/*"
!ls -lh /kaggle/working/results_tl.zip

  adding: models/ (stored 0%)
  adding: models/baseline_cnn.pth (deflated 8%)
  adding: models/resnet18_finetune.pth (deflated 7%)
  adding: models/mediumcnn.pth (deflated 7%)
  adding: models/resnet18_frozen.pth (deflated 7%)
  adding: models/medium_cnn.pth (deflated 7%)
  adding: models/medium_cnn_30ep.pth (deflated 7%)
  adding: reports/ (stored 0%)
  adding: reports/mediumcnn_history.json (deflated 57%)
  adding: reports/medium_history.json (deflated 58%)
  adding: reports/medium_30ep_history.json (deflated 60%)
  adding: reports/figures/ (stored 0%)
  adding: reports/figures/cnn_comparison.png (deflated 8%)
  adding: reports/figures/baseline_curves.png (deflated 7%)
  adding: reports/resnet18_finetune_history.json (deflated 54%)
  adding: reports/baseline_history.json (deflated 58%)
  adding: reports/resnet18_frozen_history.json (deflated 54%)
-rw-r--r-- 1 root root 100M Oct 28 09:34 /kaggle/working/results_tl.zip


In [10]:
import torch, numpy as np
from sklearn.metrics import accuracy_score, f1_score
from src.models import build_resnet18

device = 'cuda'
model = build_resnet18(num_classes=37, freeze_backbone=False)
model.load_state_dict(torch.load('models/resnet18_finetune.pth', map_location=device))
model.to(device).eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(y.numpy())

all_preds, all_labels = np.array(all_preds), np.array(all_labels)
print('test samples:', len(all_labels))
print('test accuracy:', round(accuracy_score(all_labels, all_preds), 4))
print('test macro-F1:', round(f1_score(all_labels, all_preds, average='macro'), 4))

np.save('reports/test_preds.npy', all_preds)
np.save('reports/test_labels.npy', all_labels)

test samples: 3669
test accuracy: 0.8907
test macro-F1: 0.8884


In [11]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

preds = np.load('reports/test_preds.npy')
labels = np.load('reports/test_labels.npy')

# 1) ყველაზე სუსტი 5 ჯიში (F1-ით)
report = classification_report(labels, preds, target_names=classes, output_dict=True)
per_class = sorted(((c, report[c]['f1-score']) for c in classes), key=lambda t: t[1])
print('ყველაზე სუსტი 5 ჯიში:')
for c, f1 in per_class[:5]:
    print(f'  {c:28s} F1 = {f1:.2f}')

# 2) ყველაზე ხშირად აღრეული წყვილები (ჭეშმარიტი -> პროგნოზი)
cm = confusion_matrix(labels, preds)
np.fill_diagonal(cm, 0)
pairs = sorted(((cm[i, j], classes[i], classes[j])
                for i in range(37) for j in range(37) if cm[i, j] > 0), reverse=True)
print('\nყველაზე ხშირი შეცდომები (ჭეშმარიტი -> პროგნოზი):')
for n, a, b in pairs[:8]:
    print(f'  {n:3d}  {a}  ->  {b}')

ყველაზე სუსტი 5 ჯიში:
  American Pit Bull Terrier    F1 = 0.59
  Staffordshire Bull Terrier   F1 = 0.67
  Ragdoll                      F1 = 0.73
  American Bulldog             F1 = 0.79
  Birman                       F1 = 0.82

ყველაზე ხშირი შეცდომები (ჭეშმარიტი -> პროგნოზი):
   22  American Pit Bull Terrier  ->  Staffordshire Bull Terrier
   19  Ragdoll  ->  Birman
   15  American Pit Bull Terrier  ->  American Bulldog
   14  Staffordshire Bull Terrier  ->  American Bulldog
   10  Egyptian Mau  ->  Bengal
   10  Beagle  ->  Basset Hound
    9  Russian Blue  ->  British Shorthair
    9  Basset Hound  ->  Beagle


In [12]:
!pip install -q onnx onnxruntime

import torch, time, os, numpy as np
import onnxruntime as ort
from src.models import build_resnet18

# 1) მოდელი CPU-ზე
model = build_resnet18(num_classes=37, freeze_backbone=False)
model.load_state_dict(torch.load('models/resnet18_finetune.pth', map_location='cpu'))
model.eval()

# 2) ONNX ექსპორტი
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(model, dummy, 'models/resnet18.onnx',
                  input_names=['input'], output_names=['logits'],
                  dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
                  opset_version=17, dynamo=False)

# 3) ზომა და სიჩქარე (batch=1, CPU)
sess = ort.InferenceSession('models/resnet18.onnx', providers=['CPUExecutionProvider'])
x = torch.randn(1, 3, 224, 224)

def bench(fn, n=50):
    for _ in range(5): fn()          # გახურება
    t = time.time()
    for _ in range(n): fn()
    return (time.time() - t) / n * 1000

with torch.no_grad():
    pt_ms = bench(lambda: model(x))
ox_ms = bench(lambda: sess.run(None, {'input': x.numpy()}))

# 4) შედეგები ერთნაირია?
with torch.no_grad():
    diff = np.abs(model(x).numpy() - sess.run(None, {'input': x.numpy()})[0]).max()

print(f"PyTorch : {os.path.getsize('models/resnet18_finetune.pth')/1e6:.1f} MB | {pt_ms:.1f} ms/სურათი")
print(f"ONNX    : {os.path.getsize('models/resnet18.onnx')/1e6:.1f} MB | {ox_ms:.1f} ms/სურათი")
print(f"მაქს. განსხვავება გამოსავალში: {diff:.2e}")

/tmp/ipykernel_49/1602911608.py:14: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, 'models/resnet18.onnx',


PyTorch : 44.9 MB | 33.8 ms/სურათი
ONNX    : 44.8 MB | 17.7 ms/სურათი
მაქს. განსხვავება გამოსავალში: 4.77e-07


In [13]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat

class CalibReader(CalibrationDataReader):
    def __init__(self, loader, n_batches=8):
        data = []
        for i, (x, _) in enumerate(loader):
            if i >= n_batches: break
            data.append({'input': x.numpy()})
        self.it = iter(data)
    def get_next(self):
        return next(self.it, None)

quantize_static('models/resnet18.onnx', 'models/resnet18_int8.onnx',
                CalibReader(val_loader),
                quant_format=QuantFormat.QDQ, per_channel=True,
                weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8)

# სიჩქარე და ზომა
sess_q = ort.InferenceSession('models/resnet18_int8.onnx', providers=['CPUExecutionProvider'])
q_ms = bench(lambda: sess_q.run(None, {'input': x.numpy()}))
print(f"INT8    : {os.path.getsize('models/resnet18_int8.onnx')/1e6:.1f} MB | {q_ms:.1f} ms/სურათი")

# სიზუსტე test-ზე (3669 სურათი)
preds_q = []
for xb, yb in test_loader:
    preds_q.extend(sess_q.run(None, {'input': xb.numpy()})[0].argmax(1))
preds_q = np.array(preds_q)
labels = np.load('reports/test_labels.npy')
print(f"INT8 test accuracy: {(preds_q == labels).mean():.4f}  (FP32: 0.8907)")

INT8    : 11.3 MB | 12.9 ms/სურათი
INT8 test accuracy: 0.8635  (FP32: 0.8907)


In [14]:
import json
json.dump(classes, open('models/classes.json', 'w'), ensure_ascii=False)

!cd /kaggle/working/project && zip -r /kaggle/working/final_models.zip models/resnet18_finetune.pth models/resnet18.onnx models/resnet18_int8.onnx models/classes.json reports
!ls -lh /kaggle/working/final_models.zip

  adding: models/resnet18_finetune.pth (deflated 7%)
  adding: models/resnet18.onnx (deflated 7%)
  adding: models/resnet18_int8.onnx (deflated 17%)
  adding: models/classes.json (deflated 46%)
  adding: reports/ (stored 0%)
  adding: reports/mediumcnn_history.json (deflated 57%)
  adding: reports/medium_history.json (deflated 58%)
  adding: reports/test_preds.npy (deflated 96%)
  adding: reports/test_labels.npy (deflated 99%)
  adding: reports/medium_30ep_history.json (deflated 60%)
  adding: reports/figures/ (stored 0%)
  adding: reports/figures/cnn_comparison.png (deflated 8%)
  adding: reports/figures/baseline_curves.png (deflated 7%)
  adding: reports/resnet18_finetune_history.json (deflated 54%)
  adding: reports/baseline_history.json (deflated 58%)
  adding: reports/resnet18_frozen_history.json (deflated 54%)
-rw-r--r-- 1 root root 89M Oct 28 09:46 /kaggle/working/final_models.zip
